# Card sorting sobre PRs aceptados despues de retrabajo

## Pregunta inicial

**Que tipos de problemas hacen que los pull requests generados por agentes de IA no sean aceptados de manera inmediata y requieran retrabajo antes de poder integrarse?**

Este notebook documenta el flujo usado para pasar desde el universo completo de PRs del dataset AIDev hasta una muestra revisable mediante card sorting. El foco actual son los PRs `merged_after_rework`: casos que finalmente fueron mergeados, pero solo despues de commits adicionales y comentarios humanos que permiten observar el retrabajo.

## Problema, motivacion y consecuencias

Medir solo si un PR fue mergeado no explica que ocurrio durante la revision. Un PR puede terminar aceptado y aun asi haber requerido correcciones, aclaraciones o ajustes sustantivos antes del merge.

Este flujo busca observar esa zona intermedia: contribuciones de agentes de IA que no fueron aceptadas inmediatamente, pero que si lograron integrarse despues de intervencion humana y commits adicionales.

## Enfoque metodologico: card sorting

El abordaje usa **card sorting abierto** para que las categorias emerjan desde las tarjetas, en lugar de imponer una taxonomia previa.

Pasos principales:

1. construir la poblacion `merged_after_rework`;
2. extraer una muestra estratificada por agente;
3. preparar tarjetas con evidencia textual;
4. clasificar manualmente los motivos de retrabajo;
5. analizar distribuciones por agente, lenguaje, tipo de tarea y complejidad.

## Embudo de datos

El embudo se calcula desde el resumen de muestreo generado por `sampling/stratified_sampler.py`. Los filtros poblacionales ocurren antes de estratificar.

## Universo bruto antes de construir la poblacion operacional

La tabla siguiente muestra los cortes principales antes de construir la muestra.

## Archivos usados

Las rutas vienen desde las constantes de los scripts de muestreo y preparacion. Si algun output esperado no existe o quedo vacio, la celda inicial vuelve a ejecutar el script correspondiente antes de cargar las tablas.

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "exploration" / "aidev").exists():
            return path
    raise RuntimeError("No se encontro la raiz del repositorio")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from exploration.aidev.notebook_flow import (
    build_agent_distribution,
    build_evidence_tables,
    build_files_table,
    build_funnel,
    build_outputs_flow,
    build_preparation_flow,
    build_raw_overview,
    build_template_preview,
    ensure_flow_outputs,
    validate_flow,
)
from exploration.aidev.preparation.rejection_cards import MANUAL_TEMPLATE_FIELDS
from exploration.aidev.sampling.stratified_sampler import POPULATION_MODE, STRATA_FIELDS

artifacts = ensure_flow_outputs(ROOT)
sampling_summary = artifacts.sampling_summary
preparation_summary = artifacts.preparation_summary
filter_counts = sampling_summary["population_filter_counts"]
sample_df = artifacts.sample_df
cards_df = artifacts.cards_df
template_df = artifacts.template_df

POPULATION_MODE, STRATA_FIELDS, MANUAL_TEMPLATE_FIELDS[-1], len(sample_df), len(cards_df)


('merged-after-rework', ['agent'], 'categoria_retrabajo_pre_merge', 300, 300)

## Paso 0: filtros poblacionales antes de estratificar

El corte clave es `commit_count > 1` y `human_comment_count > 0` sobre PRs ya mergeados. Esto evita mezclar rechazos definitivos con aceptaciones despues de retrabajo.

In [ ]:
build_files_table(artifacts)


In [ ]:
build_raw_overview(sampling_summary)


## Paso 1: estratificacion por agente

Una vez construida la poblacion operacional, se calcula una muestra estratificada de 300 PRs usando solo `agent` como variable de estratificacion.

In [ ]:
build_funnel(artifacts)


### Distribucion por agente

La distribucion compara poblacion, cuotas de muestreo y tarjetas finales.

In [ ]:
build_agent_distribution(artifacts)


## Paso 2: preparacion de tarjetas

Preparation recibe una muestra que ya fue filtrada por commits adicionales y comentarios humanos. El filtro `human_comment_count > 0` queda como guardia de calidad.

In [ ]:
build_preparation_flow(artifacts)


### Evidencia disponible tras preparation

La preparacion prioriza reviews, comentarios inline, comentarios generales y timeline. La tabla resume la fuente principal seleccionada para cada tarjeta.

In [ ]:
evidence, review_states = build_evidence_tables(artifacts)
display(evidence)
display(review_states)


## Paso 3: salida para card sorting

El flujo vigente produce tarjetas con evidencia y una plantilla manual para registrar categorias emergentes durante el card sorting.

In [ ]:
build_outputs_flow(artifacts)


In [ ]:
build_template_preview(artifacts)


## Validaciones de consistencia

Estas validaciones hacen explicitos los supuestos del flujo: poblacion `merged_after_rework`, estratificacion por agente, muestra de 300 PRs y una tarjeta final por PR.

In [ ]:
validate_flow(artifacts)


## Lectura metodologica

El flujo parte del universo completo de PRs, filtra antes del muestreo los casos que no muestran retrabajo observable y conserva una muestra estratificada por agente. La interpretacion cualitativa debe enfocarse en motivos de retrabajo antes del merge, no en rechazo definitivo.